We'll do a simple greenscreen replacement with a machine-learning model.

In the following code cells, we will:
- download and display an image
- pre-process the pixels, labeling background and foreground pixels
- train a model to identify green background pixels
- use the predictions of the model to swap the green screen background for an image of a forest

In [ ]:
import os
import urllib.request
import numpy as np
import tensorflow as tf
from tensorflow import keras

Download the image

In [ ]:
if not os.path.exists('../imgs/greenML.png'):
    os.makedirs('../imgs', exist_ok=True)
    urllib.request.urlretrieve('https://github.com/mlittmancs/great_courses_ml/raw/master/imgs/greenML.png', '../imgs/greenML.png')

Display the image using Keras

In [ ]:
img = keras.utils.load_img('../imgs/greenML.png')

display(img)

Display a cropped version of the image

In [ ]:
arr = keras.utils.img_to_array(img)
# Trim off edges
arr = arr[:697,:]
display(tf.keras.preprocessing.image.array_to_img(arr,scale=False))

Isolate the background and make a dataset of background pixels, `YesSet`.

In [ ]:
# background
tmp = arr[:,:360]
display(tf.keras.preprocessing.image.array_to_img(tmp,scale=False))

YesSet = np.reshape(tmp,(-1,3))

Print the number dimensions of `tmp`.

In [ ]:
tmp.shape

Print the dimensions of `YesSet`.



In [ ]:
YesSet.shape

Build `seen` dictionary of the unique pixel colors in `yes_list`.  For all keys in the dictionary, the value is 1.

In [ ]:
seen = {}
for c in YesSet:
  col = str(c)
  if col not in seen: seen[col] = 1
len(seen)

Isolate part of the foreground and make a dataset of foreground pixels called `NoSet`.

In [ ]:
# foreground
tmp = arr[30:,547:620]
display(tf.keras.preprocessing.image.array_to_img(tmp,scale=False))

NoSet = np.reshape(tmp,(-1,3))

Dimensions of the full image, corresponding to the height, width, and RGB (red, green, blue) colors.

In [ ]:
arr.shape

We finalize our dataset, with a variable `alldat` cointaining our list of pixels, and `labs` holding our list of labels for each pixel (`0` for green background pixels and `1` for foreground pixels)

In [ ]:
# Build a list of pixels for both positive and negative examples.
alldat = np.concatenate((YesSet,NoSet))
 
# labels
labs = np.concatenate((np.ones(len(YesSet)), np.zeros(len(NoSet))))

A forest background.

In [ ]:
if not os.path.exists('../imgs/forest.png'):
    os.makedirs('../imgs', exist_ok=True)
    urllib.request.urlretrieve('https://github.com/mlittmancs/great_courses_ml/raw/master/imgs/forest.jpg', '../imgs/forest.png')

We display the image of the forest, `img` and covert the image into an array, `bkg`.

In [ ]:
img = keras.utils.load_img("../imgs/forest.jpg")

display(img)

bkg = keras.utils.img_to_array(img)

Define drawScreen for merging foreground and background.

In [ ]:
import math

def distance(c1, c2):
  return(math.sqrt(  (c1[0]-c2[0])**2 + (c1[1]-c2[1])**2 + (c1[2]-c2[2])**2 ) )

def drawScreen(c, d, studio_image, background_image, ishift=157):
  i = 5
  display_image = studio_image.copy()
  for x in range(min(background_image.shape[0],studio_image.shape[0])):
    for y in range(min(background_image.shape[1], studio_image.shape[1])):
      fgi = i - ishift
      if distance(c, studio_image[x][y]) <= d:
        display_image[x][y] = background_image[x][y]
      else:
        if fgi >= 0: display_image[x][y] = studio_image[fgi][y]
  display(tf.keras.preprocessing.image.array_to_img(display_image,scale=False))

Put ML into the forest.

In [ ]:
drawScreen((0.3,174.5,46.7),math.sqrt(1000), arr, bkg)